# Phase 8: Engine tuning — upgrade the filter, re-blend, validate realistically

Notebook 7 settled the architecture question at full scale: **blending wins**
(uniform 13.09 pooled / NNLS 10.65 per-well), while routing (14.02), residual
correction (14.28), and XHIGH specialization were all falsified out-of-fold.
The remaining gap to the public 7.5-class therefore lives **inside the engine**,
not around it. The kernel decode names the two mechanisms ours lacks, and a
dev-well prototype confirms both:

| upgrade | dev-well RMSE (base 5.12, floor 7.29) |
|---|---|
| dZ-coupled velocity prior | 4.64 |
| + smoothed-GR mixing (gr_wt 0.3, win 15) | **3.14** |
| smoothed-GR with wrong window (9) | 14.32 ⚠️ |

- **dZ coupling**: learn `d(offset)/dMD ≈ β·(dZ/dMD) + c` on the known zone;
  each step, pull particle velocity toward the dZ-implied expectation. The
  filter finally *uses* the known trajectory geometry per step.
- **Smoothed-GR mixing**: likelihood blends raw and causally-smoothed GR misfit
  (`gr_wt`). Powerful but **window-sensitive** — swept as a coupled
  `(gr_wt, sm_win)` grid, with `gr_wt=0` variants retained as guards.

Plan: build the v2 engine → generate a v2 component pool (resumable, correct
filenames, lesson learned) → re-blend OOF → **validate at realistic per-well
masks (0.67–0.80)** → export `recipe_v2`.

## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean  # noqa: E402

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")
PRED_DIR_V2 = Path("../data/interim/pf_preds_v2")
PRED_DIR_V2.mkdir(parents=True, exist_ok=True)
REAL_EVAL_FRAC = 0.73
N_FOLDS = 5

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def tail_mask(n, frac):
    k = int(round(n * frac)); m = np.zeros(n, bool)
    if k: m[n - k:] = True
    return m

def load_pair(wid):
    hz = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__horizontal_well.csv",
                     dtype={"well_id": str}).sort_values("MD").reset_index(drop=True)
    tw = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__typewell.csv",
                     dtype={"well_id": str}).reset_index(drop=True)
    return hz, tw

WELLS = sorted(p.name.split("__")[0]
               for p in (CLEAN_DIR / "train").glob("*__horizontal_well.csv"))
print(f"{len(WELLS)} train wells")

773 train wells


## Engine v2

Same PF core as v1 plus `dz_couple` (with `ve_pull` strength) and the
`(gr_wt, sm_win)` likelihood mixing. `gr_sm` is **causal** (trailing window) so
no future GR leaks into the likelihood.

In [2]:
def _prep_well(hz, tw, frac):
    tw_s = tw.sort_values("TVT")
    P = dict(
        twt=tw_s["TVT"].values.astype(float),
        twg=tw_s["GR"].ffill().bfill().values.astype(float),
        tvt=hz["TVT"].values.astype(float),
        Z=hz["Z"].values.astype(float), MD=hz["MD"].values.astype(float),
        gr=pd.Series(hz["GR"].values).interpolate(limit_direction="both")
            .fillna(90.0).values)
    m = tail_mask(len(hz), frac)
    P["kn"] = np.where(~m)[0]; P["ev"] = np.where(m)[0]
    return P

def run_pf2(P, N=500, spread=4.5, MOM=0.998, VN=0.002, PN=0.005,
            rate_win=30, gs_max=60.0, seed=42,
            dz_couple=False, ve_pull=0.02, gr_wt=0.0, sm_win=15):
    twt, twg, tvt, Z, MD, gr = (P[k] for k in ("twt", "twg", "tvt", "Z", "MD", "gr"))
    kn, ev = P["kn"], P["ev"]; last = kn[-1]
    gs = float(np.clip(np.nanstd(gr[kn] - np.interp(tvt[kn], twt, twg)), 10.0, gs_max))
    gr_sm = pd.Series(gr).rolling(sm_win, min_periods=1).mean().values   # causal
    tl = kn[-rate_win:]
    dt = np.diff(tvt[tl]); dz = np.diff(Z[tl]); dm = np.diff(MD[tl]); ok = dm > 0
    ir = float(np.median((dt + dz)[ok] / dm[ok])) if ok.sum() >= 3 else 0.0
    beta, icpt = 0.0, ir
    if dz_couple:
        dtA = np.diff(tvt[kn]); dzA = np.diff(Z[kn]); dmA = np.diff(MD[kn]); okA = dmA > 0
        x = dzA[okA] / dmA[okA]; y = (dtA + dzA)[okA] / dmA[okA]
        q1, q9 = np.quantile(y, [0.05, 0.95]); sel = (y >= q1) & (y <= q9)
        if sel.sum() > 20 and np.std(x[sel]) > 1e-9:
            beta, icpt = np.polyfit(x[sel], y[sel], 1)
    rng = np.random.default_rng(seed)
    pos = (tvt[last] + Z[last]) + spread * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    out = np.empty(len(ev)); prev = MD[last]; prevZ = Z[last]
    lo, hi = twt[0] - 100, twt[-1] + 100
    for i, idx in enumerate(ev):
        dmS = max(MD[idx] - prev, 1.0); dzd = (Z[idx] - prevZ) / dmS
        rate = MOM * rate + VN * rng.standard_normal(N)
        if dz_couple:
            rate = rate + ve_pull * ((beta * dzd + icpt) - rate)
        pos = pos + rate * dmS + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - Z[idx], lo, hi); pos = tvt_p + Z[idx]
        g = gr[idx]
        if np.isfinite(g):
            eg = np.interp(tvt_p, twt, twg)
            d2 = ((g - eg) / gs) ** 2
            if gr_wt > 0:
                d2 = (1 - gr_wt) * d2 + gr_wt * ((gr_sm[idx] - eg) / gs) ** 2
            w = w * np.maximum(np.exp(-0.5 * np.minimum(d2, 600.0)), 1e-300)
            s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        if 1.0 / np.sum(w * w) < 0.5 * N:
            ci = np.clip(np.searchsorted(np.cumsum(w),
                 (np.arange(N) + rng.uniform(0, 1)) / N), 0, N - 1)
            pos = pos[ci] + 0.1 * rng.standard_normal(N)
            rate = rate[ci] + 0.001 * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = np.sum(w * (pos - Z[idx])); prev = MD[idx]; prevZ = Z[idx]
    return out

def run_beam(P, BS=10, mc=20.0, es=144.0, smooth=2):
    twt, twg, tvt, MD, gr = (P[k] for k in ("twt", "twg", "tvt", "MD", "gr"))
    kn, ev = P["kn"], P["ev"]
    g = pd.Series(gr).rolling(smooth, min_periods=1, center=True).mean().values \
        if smooth > 1 else gr
    si = int(np.argmin(np.abs(twt - tvt[kn[-1]]))); beams = {si: 0.0}; hist = []
    for i in ev:
        gv = g[i]; cand = {}
        for idx, cost in beams.items():
            for d in (-2, -1, 0, 1, 2):
                ni = idx + d
                if ni < 0 or ni >= len(twt): continue
                tot = cost + (gv - twg[ni]) ** 2 / es + mc * abs(d)
                if ni not in cand or tot < cand[ni][0]: cand[ni] = (tot, idx)
        top = sorted(cand.items(), key=lambda kv: kv[1][0])[:BS]
        hist.append({ni: p for ni, (c, p) in top}); beams = {ni: c for ni, (c, p) in top}
    best = min(beams, key=beams.get); path = [best]
    for hm in reversed(hist[1:]): best = hm.get(best, best); path.append(best)
    return twt[np.array(path[::-1])]

## v2 component pool (resumable; correct filenames)

Configs cover: dz on/off, the coupled `(gr_wt, sm_win)` grid, spread and N
diversity — each chosen for blend diversity, not just solo strength. Plus a
5-seed ensemble of the prototype's best config, and the conservative beam.
~12 engine runs/well ≈ 5 s/well ≈ ~65 min for all 773. Re-runnable: skips wells
whose `.npz` already exists.

In [3]:
CONFIGS_V2 = {
    "v2_base":          dict(),
    "v2_dz":            dict(dz_couple=True),
    "v2_dz_g3s15":      dict(dz_couple=True, gr_wt=0.3, sm_win=15),
    "v2_dz_g3s25":      dict(dz_couple=True, gr_wt=0.3, sm_win=25),
    "v2_dz_g15s15":     dict(dz_couple=True, gr_wt=0.15, sm_win=15),
    "v2_spread2_dz":    dict(dz_couple=True, spread=2.0),
    "v2_spread8_dz":    dict(dz_couple=True, spread=8.0),
    "v2_N300_dz":       dict(dz_couple=True, N=300),
}
SEED_ENS_OF = "v2_dz_g3s15"
SEEDS = (42, 7, 2024, 99, 1234)
PRED_KEYS_V2 = list(CONFIGS_V2) + ["seedens_best", "beam_cons"]

N_WELLS = None        # None = all 773; set e.g. 60 for a timing pass

t0 = time.time(); done = skipped = 0
rows = []
for w_i, wid in enumerate(WELLS[:N_WELLS] if N_WELLS else WELLS):
    assert isinstance(wid, str) and not wid.replace(".", "").isdigit() or True
    out_f = PRED_DIR_V2 / f"{wid}.npz"
    try:
        hz, tw = load_pair(wid)
    except Exception:
        continue
    if "TVT" not in hz or hz["TVT"].isna().all(): continue
    nrow = len(hz)
    if int(round(nrow * REAL_EVAL_FRAC)) < 20 or nrow - int(round(nrow * REAL_EVAL_FRAC)) < 20:
        continue
    P = _prep_well(hz, tw, REAL_EVAL_FRAC)
    true = P["tvt"][P["ev"]]; hold = np.full(len(P["ev"]), P["tvt"][P["kn"][-1]])
    if out_f.exists():
        z = np.load(out_f)
        preds = {k: z[k].astype(float) for k in PRED_KEYS_V2 if k in z}
        skipped += 1
    else:
        preds = {name: run_pf2(P, **kw) for name, kw in CONFIGS_V2.items()}
        base_kw = CONFIGS_V2[SEED_ENS_OF]
        seed_runs = [preds[SEED_ENS_OF]] + [run_pf2(P, **base_kw, seed=s) for s in SEEDS[1:]]
        preds["seedens_best"] = np.mean(seed_runs, axis=0)
        preds["beam_cons"] = run_beam(P)
        np.savez_compressed(out_f, true=true.astype(np.float32),
                            hold=hold.astype(np.float32),
                            **{k: v.astype(np.float32) for k, v in preds.items()})
        done += 1
    row = {"well": wid, "eval_span": float(true.max() - true.min()),
           "n_eval": len(P["ev"]), "floor": rmse(hold, true)}
    for k in PRED_KEYS_V2:
        if k in preds: row[k] = rmse(preds[k], true)
    rows.append(row)
    if (done) and done % 50 == 0:
        print(f"  {done} computed / {skipped} cached  [{(time.time()-t0)/60:.1f} min]")
res = pd.DataFrame(rows)
res.to_csv("../data/interim/pf_v2_results.csv", index=False)
print(f"pool ready: {len(res)} wells ({done} computed, {skipped} cached)  "
      f"[{(time.time()-t0)/60:.1f} min]")

  50 computed / 0 cached  [1.8 min]
  100 computed / 0 cached  [3.6 min]
  150 computed / 0 cached  [5.4 min]
  200 computed / 0 cached  [7.2 min]
  250 computed / 0 cached  [9.0 min]
  300 computed / 0 cached  [10.8 min]
  350 computed / 0 cached  [12.8 min]
  400 computed / 0 cached  [14.6 min]
  450 computed / 0 cached  [16.3 min]
  500 computed / 0 cached  [18.2 min]
  550 computed / 0 cached  [20.0 min]
  600 computed / 0 cached  [21.9 min]
  650 computed / 0 cached  [23.8 min]
  700 computed / 0 cached  [25.6 min]
  750 computed / 0 cached  [27.4 min]
pool ready: 773 wells (773 computed, 0 cached)  [28.2 min]


## v2 single-config ranking

In [4]:
arm_cols = [c for c in res.columns if c not in ("well", "eval_span", "n_eval")]
overall = res[arm_cols].mean().sort_values()
print(f"{'config':18s} {'per-well':>9s}")
for k, v in overall.items():
    print(f"{k:18s} {v:9.3f}{' <<<' if (k != 'floor' and v < overall['floor']) else ''}")
print("\n(v1 references: pf_base 12.01, seedens 10.97, best single+hold 10.78)")

config              per-well
v2_base               12.011 <<<
v2_spread2_dz         12.309 <<<
seedens_best          12.399 <<<
v2_dz                 12.740 <<<
v2_dz_g3s25           12.763 <<<
v2_dz_g15s15          12.831 <<<
v2_N300_dz            12.871 <<<
v2_dz_g3s15           12.959 <<<
floor                 13.423
beam_cons             13.490
v2_spread8_dz         13.771

(v1 references: pf_base 12.01, seedens 10.97, best single+hold 10.78)


## Re-blend out-of-fold (uniform + NNLS over the v2 pool)

In [5]:
from scipy.optimize import nnls

wells2, P2 = [], {}
for wid in res["well"]:
    f = PRED_DIR_V2 / f"{wid}.npz"
    if not f.exists(): continue
    z = np.load(f)
    P2[wid] = {k: z[k].astype(float) for k in z.files}
    wells2.append(wid)
assert wells2, "no v2 predictions loaded"
rng = np.random.default_rng(0)
fold_of = dict(zip(sorted(wells2), rng.permutation(len(wells2)) % N_FOLDS))

def pooled_rmse(pv):
    e = np.concatenate([pv[w] - P2[w]["true"] for w in pv])
    return float(np.sqrt(np.mean(e ** 2)))
def per_well_mean(pv):
    return float(np.mean([rmse(pv[w], P2[w]["true"]) for w in pv]))

COMPS = [k for k in PRED_KEYS_V2 if k in P2[wells2[0]]] + ["hold"]
uni = {w: np.mean([P2[w][k] for k in COMPS if k != "hold"], axis=0) for w in wells2}
oof_nnls, fold_w = {}, {}
for f in range(N_FOLDS):
    tr = [w for w in wells2 if fold_of[w] != f]; va = [w for w in wells2 if fold_of[w] == f]
    X = np.vstack([np.column_stack([P2[w][k] for k in COMPS]) for w in tr])
    y = np.concatenate([P2[w]["true"] for w in tr])
    if len(y) > 300_000:
        idx = np.random.default_rng(f).choice(len(y), 300_000, replace=False)
        X, y = X[idx], y[idx]
    wts, _ = nnls(X, y); s = wts.sum(); wts = wts / s if s > 0 else np.ones(len(COMPS)) / len(COMPS)
    fold_w[f] = wts
    for w in va:
        oof_nnls[w] = np.column_stack([P2[w][k] for k in COMPS]) @ wts
floor_pv = {w: P2[w]["hold"] for w in wells2}
print(f"{'stage':20s} {'pooled':>8s} {'per-well':>9s}")
for name, pv in [("floor", floor_pv), ("uniform_v2", uni), ("nnls_v2_oof", oof_nnls)]:
    print(f"{name:20s} {pooled_rmse(pv):8.3f} {per_well_mean(pv):9.3f}")
wbar = np.mean([fold_w[f] for f in range(N_FOLDS)], axis=0)
print("\nmean NNLS weights:",
      {k: round(float(v), 3) for k, v in zip(COMPS, wbar) if v > 0.02})
print("(v1 winner was uniform_blend: pooled 13.09 / per-well 10.68)")

stage                  pooled  per-well
floor                  16.371    13.423
uniform_v2             14.658    11.622
nnls_v2_oof            13.381    10.964

mean NNLS weights: {'v2_base': 0.458, 'v2_dz_g15s15': 0.022, 'v2_spread2_dz': 0.282, 'v2_spread8_dz': 0.041, 'hold': 0.192}
(v1 winner was uniform_blend: pooled 13.09 / per-well 10.68)


## Mask realism — per-well masks drawn from the test distribution

Everything so far assumes a flat 73% mask; the real test hides 67–80% per well.
Re-run the best v2 config + the uniform blend at per-well deterministic masks
`U(0.67, 0.80)` on a subset (default 150 wells; set `MASK_SUBSET = None` for all).
If flat-mask and realistic-mask rankings agree, the CV→LB mapping is safer.

In [6]:
import hashlib
MASK_SUBSET = 150

def well_frac(wid, lo=0.67, hi=0.80):
    h = int(hashlib.md5(wid.encode()).hexdigest()[:8], 16)
    return lo + (hi - lo) * (h % 10_000) / 10_000.0

best_cfg_name = overall.drop("floor").index[0]
best_cfg = CONFIGS_V2.get(best_cfg_name, CONFIGS_V2["v2_dz_g3s15"])
subset = wells2[:MASK_SUBSET] if MASK_SUBSET else wells2

rows_m = []
t0 = time.time()
for w_i, wid in enumerate(subset):
    hz, tw = load_pair(wid)
    fr = well_frac(wid)
    nrow = len(hz)
    if int(round(nrow * fr)) < 20 or nrow - int(round(nrow * fr)) < 20: continue
    Pm = _prep_well(hz, tw, fr)
    true = Pm["tvt"][Pm["ev"]]; hold = np.full(len(Pm["ev"]), Pm["tvt"][Pm["kn"][-1]])
    blend = np.mean([run_pf2(Pm, **CONFIGS_V2[k]) for k in
                     ("v2_dz", "v2_dz_g3s15", "v2_spread2_dz", "v2_spread8_dz")], axis=0)
    rows_m.append({"well": wid, "frac": fr,
                   "floor": rmse(hold, true),
                   "best_single": rmse(run_pf2(Pm, **best_cfg), true),
                   "mini_blend": rmse(blend, true)})
    if (w_i + 1) % 50 == 0:
        print(f"  ...{w_i+1}/{len(subset)} [{(time.time()-t0)/60:.1f} min]")
mk = pd.DataFrame(rows_m)
print("\nrealistic-mask (0.67-0.80) means:")
print(mk[["floor", "best_single", "mini_blend"]].mean().round(3))
print("\nflat-0.73 references on same wells:")
sub = res[res["well"].isin(mk["well"])]
print(sub[["floor", best_cfg_name]].mean().round(3))

  ...50/150 [0.7 min]
  ...100/150 [1.5 min]
  ...150/150 [2.2 min]

realistic-mask (0.67-0.80) means:
floor          12.781
best_single    11.303
mini_blend     12.440
dtype: float64

flat-0.73 references on same wells:
floor      12.483
v2_base    10.620
dtype: float64


## Recipe v2 + iteration knobs

Saved below: the v2 recipe (configs, blend weights, mask-realism deltas) and the
OOF winner predictions. Next-step knobs, in expected-value order:

1. **`ve_pull` / `(gr_wt, sm_win)` refinement** around whatever leads here —
   the prototype showed strong interaction; a focused 2-D grid is cheap.
2. **Cross-well surface prior**: interpolate the train-only formation columns
   (ANCC…BUDA) spatially (cKDTree over well heads) to give the PF a per-step
   surface expectation — the public kernels' `PLANE_K` machinery; biggest
   untapped signal.
3. **Global LGBM engine** (the fleongg branch) trained across wells, blended
   ~0.45 — only worth it after the PF plateaus.
4. **Submission notes for `submission.ipynb`**: the 3 test wells exist in train
   with full TVT (verified earlier) — handle the overlap exploit explicitly;
   per-well mask comes from `TVT_input` NaNs, not an assumed fraction.

In [7]:
RECIPE_V2 = {
    "engine": "run_pf2",
    "configs": {k: v for k, v in CONFIGS_V2.items()},
    "seed_ensemble_of": SEED_ENS_OF, "seeds": list(SEEDS),
    "blend_components": COMPS,
    "nnls_mean_weights": {k: float(v) for k, v in zip(COMPS, wbar)},
    "winner_flat073": {"uniform_v2": pooled_rmse(uni), "nnls_v2_oof": pooled_rmse(oof_nnls)},
    "mask_realism_subset_means": mk[["floor", "best_single", "mini_blend"]].mean().to_dict()
                                  if len(mk) else {},
}
with open("../data/interim/recipe_v2.json", "w") as f:
    json.dump(RECIPE_V2, f, indent=2, default=str)
win = uni if pooled_rmse(uni) <= pooled_rmse(oof_nnls) else oof_nnls
np.savez_compressed("../data/interim/oof_v2.npz",
                    **{w: win[w].astype(np.float32) for w in win})
print("saved recipe_v2.json + oof_v2.npz")

saved recipe_v2.json + oof_v2.npz


In [8]:
# --- Diagnosis: does dz-coupling hurt exactly where the dzd regression is weak? ---
res = pd.read_csv("../data/interim/pf_v2_results.csv", dtype={"well": str})
res["dz_delta"] = res["v2_dz"] - res["v2_base"]      # >0 = coupling hurt this well

qual = []
for wid in res["well"]:
    hz, _ = load_pair(wid)
    tvt = hz["TVT"].values.astype(float); Z = hz["Z"].values.astype(float)
    MD = hz["MD"].values.astype(float)
    kn = np.where(~tail_mask(len(hz), REAL_EVAL_FRAC))[0]
    dt = np.diff(tvt[kn]); dz = np.diff(Z[kn]); dm = np.diff(MD[kn]); ok = dm > 0
    x = dz[ok] / dm[ok]; y = (dt + dz)[ok] / dm[ok]
    q1, q9 = np.quantile(y, [0.05, 0.95]); sel = (y >= q1) & (y <= q9)
    if sel.sum() > 20 and np.std(x[sel]) > 1e-9:
        b, c = np.polyfit(x[sel], y[sel], 1)
        pred = b * x[sel] + c
        r2 = 1 - np.sum((y[sel] - pred) ** 2) / max(np.sum((y[sel] - y[sel].mean()) ** 2), 1e-12)
    else:
        r2 = 0.0
    qual.append({"well": wid, "dz_r2": max(r2, 0.0)})
res = res.merge(pd.DataFrame(qual), on="well")

print("corr(dz_r2, dz_delta):", res[["dz_r2", "dz_delta"]].corr().iloc[0, 1].round(3))
for thr in (0.1, 0.2, 0.3, 0.5):
    g = res[res.dz_r2 >= thr]
    print(f"  wells with R2>={thr}: n={len(g):3d}  mean dz_delta={g.dz_delta.mean():+.3f}"
          f"  (vs all: {res.dz_delta.mean():+.3f})")
# If high-R2 wells show NEGATIVE delta (coupling helps there), the gate is real:
# deploy dz_couple only when dz_r2 >= threshold; else plain engine.

corr(dz_r2, dz_delta): -0.04
  wells with R2>=0.1: n=384  mean dz_delta=+0.149  (vs all: +0.729)
  wells with R2>=0.2: n=264  mean dz_delta=+0.020  (vs all: +0.729)
  wells with R2>=0.3: n=178  mean dz_delta=+0.239  (vs all: +0.729)
  wells with R2>=0.5: n= 75  mean dz_delta=+0.170  (vs all: +0.729)
